# Unsloth Fine-Tuning Lab 1
**Model:** LLaMA 3.1 8B (4-bit quantized via Unsloth)  
**Task:** Supervised Fine-Tuning (SFT) with LoRA  
**Runtime:** Google Colab — T4 GPU (free tier)

> Before running: **Runtime > Change runtime type > T4 GPU**

## Step 1 — Install Dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

## Step 3 — Config

In [ ]:
MAX_SEQ_LENGTH = 2048   # Increase for longer conversations
DTYPE = None            # Auto-detect: float16 on T4, bfloat16 on A100
LOAD_IN_4BIT = True     # Reduces VRAM usage ~4x

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B"   # Unsloth-optimized model hub
OUTPUT_DIR = "outputs"

## Step 4 — Load Base Model + Tokenizer

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

## Step 5 — Add LoRA Adapters
Only adapter weights get trained (~1–3% of total parameters).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # LoRA rank — higher = more capacity, more VRAM
    target_modules=[             # Attention + MLP layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,               # Scaling factor (usually == r)
    lora_dropout=0,              # 0 is optimized in Unsloth
    bias="none",                 # "none" is optimized in Unsloth
    use_gradient_checkpointing="unsloth",  # Saves VRAM significantly
    random_state=42,
)

## Step 6 — Prompt Template
Alpaca-style instruction format. Swap for ChatML or your own format if needed.

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Required — prevents infinite generation

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_, output in zip(instructions, inputs, outputs):
        text = ALPACA_PROMPT.format(instruction, input_, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

## Step 7 — Load Dataset
Using the Alpaca-cleaned dataset (52k instruction-following examples).  
Swap `yahma/alpaca-cleaned` with any HuggingFace dataset or your own JSONL file.

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.map(format_prompts, batched=True)
print(dataset)

## Step 8 — Training Arguments

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 2 * 4 = 8
    warmup_steps=5,
    max_steps=60,                    # Quick demo — increase for real training
    # num_train_epochs=1,            # Use instead of max_steps for a full run
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",              # 8-bit optimizer saves VRAM
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir=OUTPUT_DIR,
    report_to="none",                # Change to "wandb" to log metrics
)

## Step 9 — Create Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,                   # Set True to pack short sequences — faster
    args=training_args,
)

## Step 10 — Train

In [ ]:
trainer_stats = trainer.train()
print(f"Training complete. Peak VRAM: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

## Step 11 — Inference
Test the fine-tuned model before saving.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

inputs = tokenizer(
    [ALPACA_PROMPT.format(
        "Explain what fine-tuning is in one sentence.",  # instruction
        "",                                              # input (empty ok)
        "",                                              # response (leave blank)
    )],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## Step 12 — Save Model
Choose one or more export options below.

In [ ]:
# Option A: Save LoRA adapter only (~100MB) — recommended for continued training
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
# Option B: Save merged 16-bit model — for vLLM / HuggingFace inference
# model.save_pretrained_merged("model_merged_16bit", tokenizer, save_method="merged_16bit")

In [ ]:
# Option C: Save as GGUF for Ollama / llama.cpp
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
# Option D: Push to HuggingFace Hub
# model.push_to_hub("your-username/your-model-name", token="hf_...")
# tokenizer.push_to_hub("your-username/your-model-name", token="hf_...")